### NexaCorp Chatbot

In [42]:
# import necessary libraries
import re
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
from langchain_community.document_loaders import UnstructuredWordDocumentLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings

In [1]:
from dotenv import load_dotenv

In [2]:
# load environment variables
load_dotenv()

True

Load the document

In [44]:
# load the required document
file_path = "NexaCorp_Enterprise_Policy_Handbook_v5.2.docx"
try:
    loader = UnstructuredWordDocumentLoader(
        file_path,
        mode="elements"                                 # breaks document into structured chunks based on layout
    )
    docs = loader.load()
except Exception as e:
    raise RuntimeError(f"Failed to load document: {file_path}") from e

Remove unwanted data like header, footer, table of contents

In [45]:
# use regular expressions to remove header/footer noise
def is_header_footer(text):
    NOISE_PATTERNS = [
        r"Version\s+\d+[\.\d]*\s*\|",
        r"Internal Restricted",
        r"Not for External Distribution",
        r"©\s*20\d{2}",
        r"CONFIDENTIAL",
        r"Policy.*Handbook"
    ]
    # checks if patterns exist anywhere in the text and returns True if atleast one pattern matches
    return any(re.search(p, text, re.IGNORECASE) for p in NOISE_PATTERNS)

# uses regular expression to check if line belongs to TOC or not
def is_toc_line(text):
    text = text.strip()
    return bool(re.search(
        # check for Part I/3.2+Company Overview & Mission+5
        r"^(Part\s+[IVXLC]+|[\d]+\.[\d]+)\s+.*\s+\d+$",
        text,
        re.IGNORECASE
    ))

# clean the documents by removing unwanted data
def clean_documents(docs):
    cleaned_docs = []
    skip_toc = False
    removed = 0
    for doc in docs:
        text = doc.page_content.strip()
        lower = text.lower()
        category = doc.metadata.get("category", "")
        # Remove headers/footers using metadata
        if category in ["Header", "Footer"]:
            removed += 1
            continue
        # Regex-based, if metadata failed
        if is_header_footer(text):
            removed += 1
            continue
        # TOC detection
        if "table of contents" in lower:
            skip_toc = True
            continue
        # if in TOC section and line belongs to TOC -> remove
        if skip_toc:
            if is_toc_line(text) or len(text.split()) < 3:
                continue
            else:
                skip_toc = False
        # separate table and text metadata
        if category == "Table":
            doc.metadata["type"] = "table"
        else:
            doc.metadata["type"] = "text"
        # add cleaned documents to list
        cleaned_docs.append(doc)
        
    print(f"Removed {removed} noisy chunks")
    return cleaned_docs

In [46]:
docs_clean = clean_documents(docs)

Removed 21 noisy chunks


In [47]:
# check if cleaning worked or not
print(docs[10])
print(docs_clean[10])

page_content='Tel: +1 (415) 700-9000  |  hr@nexacorp.com  |  www.nexacorp.com' metadata={'source': 'NexaCorp_Enterprise_Policy_Handbook_v5.2.docx', 'category_depth': 0, 'filename': 'NexaCorp_Enterprise_Policy_Handbook_v5.2.docx', 'last_modified': '2026-05-01T11:30:41', 'page_number': 1, 'languages': ['eng'], 'filetype': 'application/vnd.openxmlformats-officedocument.wordprocessingml.document', 'parent_id': '455472029b5887c8acdc6c539fd7b95a', 'category': 'UncategorizedText', 'element_id': '96a4b1466765e57ee816307d4291b752', 'type': 'text'}
page_content='' metadata={'source': 'NexaCorp_Enterprise_Policy_Handbook_v5.2.docx', 'languages': ['eng'], 'filename': 'NexaCorp_Enterprise_Policy_Handbook_v5.2.docx', 'filetype': 'application/vnd.openxmlformats-officedocument.wordprocessingml.document', 'category': 'PageBreak', 'element_id': '374c9b677aede328ee9f4fdefb686eb5', 'type': 'text'}


Create structured docs with section, sub-section, content 

In [48]:
structured_docs = []
current_section = "General"                             # Tracks main section 
current_subsection = ""                                 # Tracks subsection 

for doc in docs_clean:
    text = doc.page_content.strip()                     
    category = doc.metadata.get("category", "")         # Type of chunk
    
    # Skip empty chunks
    if not text:
        continue                                        
    
    # Handle titles and updates section/subsection context
    if category == "Title":
        # If title looks like "1.2", treat as subsection
        if re.match(r"\d+\.\d+", text):
            current_subsection = text
        else:
            current_section = text           # Treat as new main section
            current_subsection = ""          # Reset subsection
            
        # Add title chunk into the structured output
        structured_docs.append({
            "section": current_section,
            "subsection": current_subsection,
            "content": text,
            "type": "title"
        })
        continue                              # move to next chunk

    # Add non-title chunks to structured output
    structured_docs.append({
        "section": current_section,
        "subsection": current_subsection,
        "content": text,
        "type": category
    })

In [49]:
print(len(structured_docs))

431


Merge the documents to create parent docs

In [50]:
# Merge chunks by (section + subsection)
merged_docs = {}

for item in structured_docs:
    section = item["section"]
    subsection = item.get("subsection", "")             # default to empty if not present
    text = item["content"]
    # Create a unique identifier for grouping
    key = f"{section} > {subsection}"   
    # Initialize group if not already present
    if key not in merged_docs:
        merged_docs[key] = {
            "section": section,
            "subsection": subsection,
            "content": ""
        }
    # Append the current chunk’s text to the grouped content
    merged_docs[key]["content"] += "\n" + text

In [51]:
print(len(merged_docs))

150


In [52]:
# convert chunks to LangChain Documents
parent_docs = []

for item in merged_docs.values():
    parent_docs.append(
        Document(
            page_content=item["content"].strip(),
            metadata={
                "section": item["section"],
                "subsection": item["subsection"]
            }
        )
    )

In [53]:
# create embeddings
embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-base-en-v1.5"
)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1499.08it/s]


In [54]:
# create FAISS vector store
import faiss
from langchain_community.vectorstores import FAISS
from langchain_community.docstore.in_memory import InMemoryDocstore

index = faiss.IndexFlatL2(768)

docstore = InMemoryDocstore({})
vectorstore = FAISS(
    embedding_function=embeddings,
    index=index,
    docstore=docstore,
    index_to_docstore_id={}
)

Parent Document Retriever

In [55]:
from langchain_classic.retrievers import ParentDocumentRetriever
from langchain_core.stores import InMemoryStore
from langchain_text_splitters import RecursiveCharacterTextSplitter

# child chunks 
child_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=100
)

# parent chunks 
parent_splitter = RecursiveCharacterTextSplitter(
    chunk_size=2000,
    chunk_overlap=300
)

store = InMemoryStore()

parent_retriever = ParentDocumentRetriever(
    vectorstore=vectorstore,   
    docstore=store,
    child_splitter=child_splitter,
    parent_splitter=parent_splitter,
)

In [56]:
parent_retriever.add_documents(parent_docs)

In [57]:
docs = parent_retriever.invoke("leave policy")

In [58]:
print(docs)

[Document(metadata={'section': '3. Human Resources Policies', 'subsection': '3.7 Leave Management Policy'}, page_content='3.7 Leave Management Policy'), Document(metadata={'section': '3. Human Resources Policies', 'subsection': '3.7.3 Leave of Absence Types'}, page_content='3.7.3 Leave of Absence Types\nIn addition to PTO and holidays, NexaCorp recognises the following formal leave types:\nLeave Type Max Duration Paid? Notice Required Documentation FMLA / Medical Leave 12 weeks/year Partially (STD integration) 30 days or ASAP Medical certification Parental Leave See Section 3.4.5 Yes — 100% 90 days preferred Birth/adoption record Military Leave (USERRA) Indefinite job protection 14 days paid; USERRA thereafter Advance notice required Military orders Jury Duty Leave Duration of service Full pay to 10 days; unpaid thereafter Summons receipt Jury summons copy Bereavement Leave 3–5 days Yes — 100% Same day Obituary/death cert. optional Volunteer Leave 3 days/year Yes — 100% 5 business days

In [59]:
from langchain_classic.retrievers.multi_query import MultiQueryRetriever

llm = init_chat_model("google_genai:gemini-2.5-flash")

multi_retriever = MultiQueryRetriever.from_llm(
    retriever=parent_retriever,
    llm=llm
)

In [60]:
from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import LLMChainExtractor

compressor = LLMChainExtractor.from_llm(llm)

final_retriever = ContextualCompressionRetriever(
    base_retriever=multi_retriever,
    base_compressor=compressor
)

In [61]:
query = "Explain company values and organisational structure together"
docs = final_retriever.invoke(query)
for i, doc in enumerate(docs):
    print(f"\n--- Chunk {i+1} ---")
    print("Section:", doc.metadata.get("section"))
    print("Content:", doc.page_content)


--- Chunk 1 ---
Section: 1. Company Overview & Mission
Content: 1.3 Organisational Structure

--- Chunk 2 ---
Section: 1. Company Overview & Mission
Content: 1.2.3 Core Values
Value What It Means in Practice Integrity First We act honestly and transparently even when it is difficult. We do not misrepresent data, timelines, or capabilities to clients, colleagues, or regulators. Customer Obsession Every decision is evaluated through the lens of long-term customer value. Short-term revenue that compromises customer trust is never acceptable. Bold Innovation We reward calculated risk-taking and intelligent failure. Bureaucracy that impedes experimentation must be challenged through proper channels. Inclusive Excellence Diversity is not a compliance exercise — it is a competitive advantage. We build teams that reflect the global communities we serve. Ownership Mindset Every employee takes responsibility for outcomes, not just activities. We do not hide behind process when results fall shor

In [62]:
query = "Explain company values and organisational structure together"
docs = parent_retriever.invoke(query)
for i, doc in enumerate(docs):
    print(f"\n--- Chunk {i+1} ---")
    print("Section:", doc.metadata.get("section"))
    print("Content:", doc.page_content)


--- Chunk 1 ---
Section: 1. Company Overview & Mission
Content: 1.3 Organisational Structure

--- Chunk 2 ---
Section: 1. Company Overview & Mission
Content: 1. Company Overview & Mission

--- Chunk 3 ---
Section: 1. Company Overview & Mission
Content: 1.2.3 Core Values
Value What It Means in Practice Integrity First We act honestly and transparently even when it is difficult. We do not misrepresent data, timelines, or capabilities to clients, colleagues, or regulators. Customer Obsession Every decision is evaluated through the lens of long-term customer value. Short-term revenue that compromises customer trust is never acceptable. Bold Innovation We reward calculated risk-taking and intelligent failure. Bureaucracy that impedes experimentation must be challenged through proper channels. Inclusive Excellence Diversity is not a compliance exercise — it is a competitive advantage. We build teams that reflect the global communities we serve. Ownership Mindset Every employee takes respons

## QUERY DECOMPOSITION

In [1]:
# check if decomposition is needed or not 
def needs_decomposition(query: str):
    query = query.lower()
    triggers = [" and ", " or ", ",", " vs ", "compare"]
    return any(t in query for t in triggers)

# rule-based decomposition
def decompose_rule_based(query: str):
    query = query.lower()
    if " and " in query:
        return [q.strip() for q in query.split(" and ")]
    if "," in query:
        return [q.strip() for q in query.split(",")]
    return [query]

# LLM-based decomposition
llm = init_chat_model("google_genai:gemini-2.5-flash-lite")
def decompose_query_llm(query):
    prompt = f"""
        Break this query into smaller independent search queries.

        Query: {query}

        Rules:
        - Max 3 queries
        - Keep them short
        - Return ONLY JSON array

        Example:
        ["company values", "organizational structure"]
        """
    try:
        response = llm.invoke(prompt)
        queries = json.loads(response.content)
        if isinstance(queries, list) and queries:
            return queries[:3]
    except:
        pass
    return [query]

# generate sub-queries based on user_query
def get_sub_queries(query):
    # check if needed
    if not needs_decomposition(query):
        return [query]
    # try rule-based
    rule_queries = decompose_rule_based(query)
    if len(rule_queries) > 1:
        return rule_queries
    # fallback to LLM
    return decompose_query_llm(query)

def retrieve(query):
    sub_queries = get_sub_queries(query)

    all_docs = []

    for q in sub_queries:
        docs = compression_retriever.invoke(q)
        all_docs.extend(docs)

    return deduplicate_docs(all_docs)

NameError: name 'init_chat_model' is not defined

In [ ]:
def build_structured_docs(docs_clean):
    structured_docs = []
    current_part = ""
    current_section = ""
    current_subsection = ""
    buffer = []

    def flush_buffer(doc_type="text"):
        if buffer:
            structured_docs.append({
                "part": current_part,
                "section": current_section,
                "subsection": current_subsection,
                "content": " ".join(buffer).strip(),
                "type": doc_type
            })
            buffer.clear()

    for doc in docs_clean:
        text = re.sub(r"\s+", " ", doc.page_content).strip()
        category = doc.metadata.get("category", "")
        
        if not text:
            continue

        # HEADING DETECTION
        if category == "Title":
            # save previous content
            flush_buffer()

            # LEVEL 3
            if re.match(r"^\d+\.\d+\.\d+\s+", text):
                current_subsection = text

            # LEVEL 2
            elif re.match(r"^\d+\.\d+\s+", text):
                current_section = text
                current_subsection = ""

            # LEVEL 1
            elif re.match(r"^\d+\.\s+", text):
                current_part = text
                current_section = ""
                current_subsection = ""
            continue

        # TABLE HANDLING
        if (
            doc.metadata.get("type") == "table"
            or category == "Table"
        ):
            flush_buffer()
            structured_docs.append({
                "part": current_part,
                "section": current_section,
                "subsection": current_subsection,
                "content": text,
                "type": "table"
            })
            continue

        # NORMAL TEXT
        buffer.append(text)

    # final 
    flush_buffer()
    print(f"Created {len(structured_docs)} structured documents")
    return structured_docs